# Parcial para la casa · Raíces B — ¿Cuándo cruzará el CO₂ las 450 ppm?
**Métodos Numéricos · UNAL Manizales · Punto fijo y secante con datos reales de Mauna Loa**

Ritual de siempre: **predice en tu guía → corre la celda → explica la diferencia**. Solo cambia lo marcado `# <-- CAMBIA ESTO`.

**Fuente de los datos.** Promedios mensuales de CO₂ (ppm, fracción molar en aire seco) medidos en el Observatorio de Mauna Loa, Hawái.
NOAA Global Monitoring Laboratory (GML) y Scripps Institution of Oceanography (curva de Keeling).
Archivo: https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv · Página: https://gml.noaa.gov/ccgg/trends/

- **Fecha de consulta:** 15 de octubre de 2026.
- **Licencia/uso:** datos públicos del gobierno de EE. UU.; uso libre citando a NOAA GML y Scripps. El respaldo congelado viene del espejo https://github.com/datasets/co2-ppm (licencia ODC-PDDL, dominio público), que copia el archivo de NOAA (comprobado: septiembre de 2025 = 424,37 ppm en ambos).
- **Corte común:** usamos datos hasta **agosto de 2026** (t = 2026,625) para que todos tengamos los mismos números. NOAA marca el último año como *preliminar*: si tu pantalla difiere en el último decimal, usa tus números.

In [ ]:
# Celda 1 · Cargar los datos (de NOAA en vivo, o del respaldo congelado si falla)
import io, socket
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
socket.setdefaulttimeout(20)

URL = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv"
CORTE = 2026.625          # agosto de 2026 (último mes del respaldo). No lo cambies.
USAR_RESPALDO = False     # pon True si quieres trabajar sin internet

RESPALDO = """1958.2027,315.71
1958.2877,317.45
1958.3699,317.51
1958.4548,317.27
1958.5370,315.87
1958.6219,314.93
1958.7068,313.21
1958.7890,312.42
1958.8740,313.33
1958.9562,314.67
1959.0411,315.58
1959.1260,316.49
1959.2027,316.65
1959.2877,317.72
1959.3699,318.29
1959.4548,318.15
1959.5370,316.54
1959.6219,314.80
1959.7068,313.84
1959.7890,313.33
1959.8740,314.81
1959.9562,315.58
1960.0410,316.43
1960.1257,316.98
1960.2049,317.58
1960.2896,319.03
1960.3716,320.03
1960.4563,319.58
1960.5383,318.18
1960.6230,315.90
1960.7077,314.17
1960.7896,313.83
1960.8743,315.00
1960.9563,316.19
1961.0411,316.89
1961.1260,317.70
1961.2027,318.54
1961.2877,319.48
1961.3699,320.58
1961.4548,319.77
1961.5370,318.56
1961.6219,316.79
1961.7068,314.99
1961.7890,315.31
1961.8740,316.10
1961.9562,317.01
1962.0411,317.94
1962.1260,318.55
1962.2027,319.68
1962.2877,320.57
1962.3699,321.02
1962.4548,320.62
1962.5370,319.61
1962.6219,317.40
1962.7068,316.24
1962.7890,315.42
1962.8740,316.69
1962.9562,317.70
1963.0411,318.74
1963.1260,319.07
1963.2027,319.86
1963.2877,321.38
1963.3699,322.24
1963.4548,321.49
1963.5370,319.74
1963.6219,317.77
1963.7068,316.21
1963.7890,315.99
1963.8740,317.07
1963.9562,318.35
1964.0410,319.57
1964.1257,320.04
1964.2049,320.75
1964.2896,321.84
1964.3716,322.25
1964.4563,321.89
1964.5383,320.44
1964.6230,318.69
1964.7077,316.71
1964.7896,316.87
1964.8743,317.68
1964.9563,318.71
1965.0411,319.44
1965.1260,320.44
1965.2027,320.89
1965.2877,322.14
1965.3699,322.17
1965.4548,321.87
1965.5370,321.21
1965.6219,318.87
1965.7068,317.82
1965.7890,317.30
1965.8740,318.87
1965.9562,319.42
1966.0411,320.62
1966.1260,321.60
1966.2027,322.39
1966.2877,323.70
1966.3699,324.08
1966.4548,323.75
1966.5370,322.37
1966.6219,320.36
1966.7068,318.64
1966.7890,318.10
1966.8740,319.78
1966.9562,321.02
1967.0411,322.33
1967.1260,322.50
1967.2027,323.03
1967.2877,324.41
1967.3699,325.00
1967.4548,324.09
1967.5370,322.54
1967.6219,320.92
1967.7068,319.25
1967.7890,319.39
1967.8740,320.73
1967.9562,321.96
1968.0410,322.57
1968.1257,323.15
1968.2049,323.89
1968.2896,325.02
1968.3716,325.57
1968.4563,325.36
1968.5383,324.14
1968.6230,322.11
1968.7077,320.33
1968.7896,320.25
1968.8743,321.32
1968.9563,322.89
1969.0411,324.00
1969.1260,324.41
1969.2027,325.63
1969.2877,326.66
1969.3699,327.38
1969.4548,326.71
1969.5370,325.88
1969.6219,323.66
1969.7068,322.38
1969.7890,321.78
1969.8740,322.85
1969.9562,324.11
1970.0411,325.06
1970.1260,325.99
1970.2027,326.93
1970.2877,328.13
1970.3699,328.08
1970.4548,327.67
1970.5370,326.34
1970.6219,324.68
1970.7068,323.10
1970.7890,323.06
1970.8740,324.01
1970.9562,325.13
1971.0411,326.17
1971.1260,326.69
1971.2027,327.18
1971.2877,327.78
1971.3699,328.93
1971.4548,328.57
1971.5370,327.36
1971.6219,325.43
1971.7068,323.36
1971.7890,323.56
1971.8740,324.80
1971.9562,326.01
1972.0410,326.77
1972.1257,327.63
1972.2049,327.75
1972.2896,329.72
1972.3716,330.07
1972.4563,329.09
1972.5383,328.04
1972.6230,326.32
1972.7077,324.84
1972.7896,325.20
1972.8743,326.50
1972.9563,327.55
1973.0411,328.55
1973.1260,329.57
1973.2027,330.30
1973.2877,331.50
1973.3699,332.48
1973.4548,332.07
1973.5370,330.87
1973.6219,329.31
1973.7068,327.52
1973.7890,327.19
1973.8740,328.17
1973.9562,328.65
1974.0411,329.36
1974.1260,330.71
1974.2027,331.49
1974.2877,332.65
1974.3750,333.19
1974.4583,332.20
1974.5417,331.07
1974.6250,329.15
1974.7083,327.33
1974.7917,327.28
1974.8750,328.31
1974.9583,329.58
1975.0417,330.73
1975.1250,331.46
1975.2083,331.94
1975.2917,333.11
1975.3750,333.95
1975.4583,333.42
1975.5417,331.97
1975.6250,329.95
1975.7083,328.49
1975.7917,328.36
1975.8750,329.38
1975.9583,330.77
1976.0417,331.56
1976.1250,332.74
1976.2083,333.36
1976.2917,334.74
1976.3750,334.72
1976.4583,333.97
1976.5417,333.08
1976.6250,330.68
1976.7083,328.96
1976.7917,328.72
1976.8750,330.16
1976.9583,331.62
1977.0417,332.68
1977.1250,333.17
1977.2083,334.96
1977.2917,336.14
1977.3750,336.93
1977.4583,336.17
1977.5417,334.89
1977.6250,332.56
1977.7083,331.29
1977.7917,331.28
1977.8750,332.46
1977.9583,333.60
1978.0417,334.94
1978.1250,335.26
1978.2083,336.66
1978.2917,337.69
1978.3750,338.02
1978.4583,338.01
1978.5417,336.50
1978.6250,334.42
1978.7083,332.36
1978.7917,332.45
1978.8750,333.76
1978.9583,334.91
1979.0417,336.14
1979.1250,336.69
1979.2083,338.27
1979.2917,338.82
1979.3750,339.24
1979.4583,339.26
1979.5417,337.54
1979.6250,335.72
1979.7083,333.97
1979.7917,334.24
1979.8750,335.32
1979.9583,336.81
1980.0417,337.90
1980.1250,338.34
1980.2083,340.07
1980.2917,340.93
1980.3750,341.45
1980.4583,341.36
1980.5417,339.45
1980.6250,337.67
1980.7083,336.25
1980.7917,336.14
1980.8750,337.30
1980.9583,338.29
1981.0417,339.29
1981.1250,340.55
1981.2083,341.63
1981.2917,342.60
1981.3750,343.04
1981.4583,342.54
1981.5417,340.82
1981.6250,338.48
1981.7083,336.95
1981.7917,337.05
1981.8750,338.58
1981.9583,339.91
1982.0417,340.93
1982.1250,341.76
1982.2083,342.78
1982.2917,343.96
1982.3750,344.77
1982.4583,343.88
1982.5417,342.42
1982.6250,340.24
1982.7083,338.37
1982.7917,338.41
1982.8750,339.44
1982.9583,340.78
1983.0417,341.57
1983.1250,342.78
1983.2083,343.37
1983.2917,345.40
1983.3750,346.14
1983.4583,345.76
1983.5417,344.32
1983.6250,342.51
1983.7083,340.46
1983.7917,340.53
1983.8750,341.79
1983.9583,343.20
1984.0417,344.21
1984.1250,344.92
1984.2083,345.68
1984.2917,347.37
1984.3750,347.77
1984.4583,347.16
1984.5417,345.79
1984.6250,343.74
1984.7083,341.59
1984.7917,341.86
1984.8750,343.31
1984.9583,345.00
1985.0417,345.48
1985.1250,346.41
1985.2083,347.91
1985.2917,348.66
1985.3750,349.28
1985.4583,348.65
1985.5417,346.90
1985.6250,345.26
1985.7083,343.47
1985.7917,343.35
1985.8750,344.73
1985.9583,346.12
1986.0417,346.78
1986.1250,347.48
1986.2083,348.25
1986.2917,349.86
1986.3750,350.52
1986.4583,349.98
1986.5417,348.25
1986.6250,346.17
1986.7083,345.48
1986.7917,344.82
1986.8750,346.22
1986.9583,347.48
1987.0417,348.73
1987.1250,348.92
1987.2083,349.81
1987.2917,351.40
1987.3750,352.15
1987.4583,351.58
1987.5417,350.21
1987.6250,348.20
1987.7083,346.66
1987.7917,346.72
1987.8750,348.08
1987.9583,349.28
1988.0417,350.51
1988.1250,351.70
1988.2083,352.50
1988.2917,353.67
1988.3750,354.35
1988.4583,353.88
1988.5417,352.80
1988.6250,350.49
1988.7083,348.97
1988.7917,349.37
1988.8750,350.42
1988.9583,351.62
1989.0417,353.07
1989.1250,353.43
1989.2083,354.08
1989.2917,355.72
1989.3750,355.95
1989.4583,355.44
1989.5417,354.05
1989.6250,351.84
1989.7083,350.09
1989.7917,350.33
1989.8750,351.55
1989.9583,352.91
1990.0417,353.86
1990.1250,355.10
1990.2083,355.75
1990.2917,356.38
1990.3750,357.38
1990.4583,356.39
1990.5417,354.89
1990.6250,353.06
1990.7083,351.38
1990.7917,351.69
1990.8750,353.14
1990.9583,354.41
1991.0417,354.93
1991.1250,355.82
1991.2083,357.33
1991.2917,358.77
1991.3750,359.23
1991.4583,358.23
1991.5417,356.30
1991.6250,353.97
1991.7083,352.34
1991.7917,352.43
1991.8750,353.89
1991.9583,355.21
1992.0417,356.34
1992.1250,357.21
1992.2083,357.97
1992.2917,359.22
1992.3750,359.71
1992.4583,359.44
1992.5417,357.15
1992.6250,354.99
1992.7083,353.01
1992.7917,353.40
1992.8750,354.42
1992.9583,355.68
1993.0417,357.10
1993.1250,357.42
1993.2083,358.59
1993.2917,359.39
1993.3750,360.30
1993.4583,359.64
1993.5417,357.45
1993.6250,355.76
1993.7083,354.14
1993.7917,354.23
1993.8750,355.53
1993.9583,357.03
1994.0417,358.36
1994.1250,359.04
1994.2083,360.11
1994.2917,361.36
1994.3750,361.78
1994.4583,360.94
1994.5417,359.51
1994.6250,357.59
1994.7083,355.86
1994.7917,356.21
1994.8750,357.65
1994.9583,359.10
1995.0417,360.04
1995.1250,361.00
1995.2083,361.98
1995.2917,363.44
1995.3750,363.83
1995.4583,363.33
1995.5417,361.78
1995.6250,359.33
1995.7083,358.32
1995.7917,358.14
1995.8750,359.61
1995.9583,360.82
1996.0417,362.20
1996.1250,363.36
1996.2083,364.28
1996.2917,364.69
1996.3750,365.25
1996.4583,365.06
1996.5417,363.69
1996.6250,361.55
1996.7083,359.69
1996.7917,359.72
1996.8750,361.04
1996.9583,362.39
1997.0417,363.24
1997.1250,364.21
1997.2083,364.65
1997.2917,366.49
1997.3750,366.77
1997.4583,365.73
1997.5417,364.46
1997.6250,362.40
1997.7083,360.44
1997.7917,360.97
1997.8750,362.65
1997.9583,364.51
1998.0417,365.39
1998.1250,366.10
1998.2083,367.36
1998.2917,368.79
1998.3750,369.56
1998.4583,369.13
1998.5417,367.98
1998.6250,366.10
1998.7083,364.16
1998.7917,364.54
1998.8750,365.67
1998.9583,367.30
1999.0417,368.35
1999.1250,369.28
1999.2083,369.84
1999.2917,371.15
1999.3750,371.12
1999.4583,370.46
1999.5417,369.61
1999.6250,367.06
1999.7083,364.95
1999.7917,365.52
1999.8750,366.88
1999.9583,368.26
2000.0417,369.45
2000.1250,369.71
2000.2083,370.75
2000.2917,371.98
2000.3750,371.75
2000.4583,371.87
2000.5417,370.02
2000.6250,368.27
2000.7083,367.15
2000.7917,367.18
2000.8750,368.53
2000.9583,369.83
2001.0417,370.76
2001.1250,371.69
2001.2083,372.63
2001.2917,373.55
2001.3750,374.03
2001.4583,373.40
2001.5417,371.68
2001.6250,369.78
2001.7083,368.34
2001.7917,368.61
2001.8750,369.94
2001.9583,371.42
2002.0417,372.70
2002.1250,373.37
2002.2083,374.30
2002.2917,375.19
2002.3750,375.93
2002.4583,375.69
2002.5417,374.16
2002.6250,372.03
2002.7083,370.92
2002.7917,370.73
2002.8750,372.43
2002.9583,373.98
2003.0417,375.07
2003.1250,375.82
2003.2083,376.64
2003.2917,377.92
2003.3750,378.78
2003.4583,378.46
2003.5417,376.88
2003.6250,374.57
2003.7083,373.34
2003.7917,373.31
2003.8750,374.84
2003.9583,376.17
2004.0417,377.17
2004.1250,378.05
2004.2083,379.06
2004.2917,380.54
2004.3750,380.80
2004.4583,379.87
2004.5417,377.65
2004.6250,376.17
2004.7083,374.43
2004.7917,374.63
2004.8750,376.33
2004.9583,377.68
2005.0417,378.63
2005.1250,379.91
2005.2083,380.95
2005.2917,382.48
2005.3750,382.64
2005.4583,382.40
2005.5417,380.93
2005.6250,378.93
2005.7083,376.89
2005.7917,377.19
2005.8750,378.54
2005.9583,380.31
2006.0417,381.58
2006.1250,382.40
2006.2083,382.86
2006.2917,384.80
2006.3750,385.22
2006.4583,384.24
2006.5417,382.65
2006.6250,380.60
2006.7083,379.04
2006.7917,379.33
2006.8750,380.35
2006.9583,382.02
2007.0417,383.10
2007.1250,384.12
2007.2083,384.81
2007.2917,386.73
2007.3750,386.78
2007.4583,386.33
2007.5417,384.73
2007.6250,382.24
2007.7083,381.20
2007.7917,381.37
2007.8750,382.70
2007.9583,384.19
2008.0417,385.78
2008.1250,386.06
2008.2083,386.28
2008.2917,387.33
2008.3750,388.78
2008.4583,387.99
2008.5417,386.62
2008.6250,384.32
2008.7083,383.41
2008.7917,383.22
2008.8750,384.41
2008.9583,385.79
2009.0417,387.17
2009.1250,387.70
2009.2083,389.04
2009.2917,389.76
2009.3750,390.36
2009.4583,389.70
2009.5417,388.24
2009.6250,386.29
2009.7083,384.95
2009.7917,384.64
2009.8750,386.23
2009.9583,387.63
2010.0417,388.91
2010.1250,390.41
2010.2083,391.37
2010.2917,392.67
2010.3750,393.21
2010.4583,392.38
2010.5417,390.41
2010.6250,388.54
2010.7083,387.03
2010.7917,387.43
2010.8750,388.87
2010.9583,389.99
2011.0417,391.50
2011.1250,392.05
2011.2083,392.80
2011.2917,393.44
2011.3750,394.41
2011.4583,393.95
2011.5417,392.72
2011.6250,390.33
2011.7083,389.28
2011.7917,389.19
2011.8750,390.48
2011.9583,392.06
2012.0417,393.31
2012.1250,394.04
2012.2083,394.59
2012.2917,396.38
2012.3750,396.93
2012.4583,395.91
2012.5417,394.56
2012.6250,392.59
2012.7083,391.32
2012.7917,391.27
2012.8750,393.20
2012.9583,394.57
2013.0417,395.78
2013.1250,397.03
2013.2083,397.66
2013.2917,398.64
2013.3750,400.02
2013.4583,398.81
2013.5417,397.51
2013.6250,395.39
2013.7083,393.72
2013.7917,393.90
2013.8750,395.36
2013.9583,397.03
2014.0417,398.04
2014.1250,398.27
2014.2083,399.91
2014.2917,401.51
2014.3750,401.96
2014.4583,401.43
2014.5417,399.27
2014.6250,397.18
2014.7083,395.54
2014.7917,396.16
2014.8750,397.40
2014.9583,399.08
2015.0417,400.18
2015.1250,400.55
2015.2083,401.74
2015.2917,403.35
2015.3750,404.15
2015.4583,402.97
2015.5417,401.46
2015.6250,399.11
2015.7083,397.82
2015.7917,398.49
2015.8750,400.27
2015.9583,402.06
2016.0417,402.73
2016.1250,404.25
2016.2083,405.06
2016.2917,407.60
2016.3750,407.90
2016.4583,406.99
2016.5417,404.59
2016.6250,402.45
2016.7083,401.23
2016.7917,401.79
2016.8750,403.72
2016.9583,404.64
2017.0417,406.36
2017.1250,406.66
2017.2083,407.54
2017.2917,409.22
2017.3750,409.89
2017.4583,409.08
2017.5417,407.33
2017.6250,405.32
2017.7083,403.57
2017.7917,403.82
2017.8750,405.31
2017.9583,407.00
2018.0417,408.15
2018.1250,408.52
2018.2083,409.59
2018.2917,410.45
2018.3750,411.44
2018.4583,410.99
2018.5417,408.90
2018.6250,407.16
2018.7083,405.71
2018.7917,406.19
2018.8750,408.21
2018.9583,409.27
2019.0417,411.03
2019.1250,411.96
2019.2083,412.18
2019.2917,413.54
2019.3750,414.86
2019.4583,414.15
2019.5417,411.96
2019.6250,410.17
2019.7083,408.76
2019.7917,408.74
2019.8750,410.47
2019.9583,411.97
2020.0417,413.59
2020.1250,414.32
2020.2083,414.72
2020.2917,416.42
2020.3750,417.28
2020.4583,416.58
2020.5417,414.58
2020.6250,412.75
2020.7083,411.50
2020.7917,411.49
2020.8750,413.10
2020.9583,414.23
2021.0417,415.49
2021.1250,416.72
2021.2083,417.61
2021.2917,419.01
2021.3750,419.09
2021.4583,418.93
2021.5417,416.90
2021.6250,414.42
2021.7083,413.26
2021.7917,413.90
2021.8750,414.97
2021.9583,416.67
2022.0417,418.13
2022.1250,419.24
2022.2083,418.76
2022.2917,420.19
2022.3750,420.97
2022.4583,420.94
2022.5417,418.85
2022.6250,417.15
2022.7083,415.91
2022.7917,415.74
2022.8750,417.47
2022.9583,418.99
2023.0417,419.48
2023.1250,420.31
2023.2083,420.99
2023.2917,423.35
2023.3750,424.00
2023.4583,423.68
2023.5417,421.83
2023.6250,419.68
2023.7083,418.50
2023.7917,418.82
2023.8750,420.46
2023.9583,421.86
2024.0417,422.80
2024.1250,424.55
2024.2083,425.38
2024.2917,426.51
2024.3750,426.90
2024.4583,426.91
2024.5417,425.55
2024.6250,422.99
2024.7083,422.03
2024.7917,422.38
2024.8750,423.85
2024.9583,425.40
2025.0417,426.65
2025.1250,427.09
2025.2083,428.15
2025.2917,429.64
2025.3750,430.51
2025.4583,429.61
2025.5417,427.87
2025.6250,425.48
2025.7083,424.37
2025.7917,424.87
2025.8750,426.46
2025.9583,427.49
2026.0417,428.62
2026.1250,429.35
2026.2083,430.15
2026.2917,431.12
2026.3750,432.34
2026.4583,431.43
2026.5417,429.13
2026.6250,427.55"""   # año decimal, CO2 mensual (ppm), 1958.2027 a 2026.6250
respaldo = pd.read_csv(io.StringIO(RESPALDO), names=["t", "co2"])

try:
    if USAR_RESPALDO:
        raise RuntimeError("se pidió el respaldo")
    crudo = pd.read_csv(URL, comment="#")
    crudo.columns = [str(s).strip().lower() for s in crudo.columns]
    if "decimal date" in crudo.columns and "average" in crudo.columns:
        vivo = pd.DataFrame({"t": crudo["decimal date"], "co2": crudo["average"]})
    else:  # por si NOAA cambia los nombres: columnas 3 y 4 (año decimal, promedio)
        vivo = pd.DataFrame({"t": crudo.iloc[:, 2], "co2": crudo.iloc[:, 3]})
    vivo = vivo.apply(pd.to_numeric, errors="coerce").dropna()
    if len(vivo) < 500 or vivo.t.max() < 2026:   # algo raro llegó: mejor el respaldo
        raise ValueError("archivo incompleto")
    datos, origen = vivo, "NOAA en vivo"
except Exception as e:
    datos = respaldo.copy()
    origen = "respaldo congelado (" + type(e).__name__ + ")"

datos = datos[(datos.co2 > 0) & (datos.t <= CORTE + 1e-6)].reset_index(drop=True)
print("Origen de los datos:", origen)
print("Meses:", len(datos), "| desde", datos.t.iloc[0], "hasta", datos.t.iloc[-1])
print("Último dato: t =", datos.t.iloc[-1], " CO2 =", datos.co2.iloc[-1], "ppm")
for anio in (2023, 2024, 2025):
    m = datos[(datos.t >= anio) & (datos.t < anio + 1)].co2.mean()
    print(f"Promedio anual {anio}: {m:.2f} ppm")
# Comparación con el respaldo (debería dar ~0)
comun = datos.merge(respaldo, on="t", suffixes=("", "_resp"))
print("Máxima diferencia con el respaldo:", round(float((comun.co2 - comun.co2_resp).abs().max()), 3), "ppm")

## El modelo (caja negra)
Ajustamos a los datos desde 1990 el modelo

c(t) = a₀ + a₁·(t − 2000) + a₂·(t − 2000)² + A·sen(2πt) + B·cos(2πt)

una **parábola** (la tendencia, que se acelera) más una **onda de periodo 1 año** (la "respiración" de la vegetación del hemisferio norte: el CO₂ sube en invierno y baja en verano).
Los cinco números a₀, a₁, a₂, A, B los escoge `numpy.linalg.lstsq` de modo que la curva quede lo más cerca posible de los datos (*mínimos cuadrados*; no necesitas saber cómo lo hace).
Usamos t − 2000 en vez de t para no elevar al cuadrado números como 2026 (¿recuerdas la cancelación?).

⚠️ **El cruce que vamos a calcular es el del MODELO ajustado, no una predicción climática oficial.**

In [ ]:
# Celda 2 · Ajuste del modelo (caja negra) y gráfica
AÑO_INICIO = 1990   # <-- CAMBIA ESTO solo en el ítem II.6 (ventana de datos usada en el ajuste)

def ajustar(anio_inicio):
    d = datos[datos.t >= anio_inicio]
    t, y = d.t.values, d.co2.values
    X = np.column_stack([np.ones_like(t), t - 2000, (t - 2000)**2,
                         np.sin(2*np.pi*t), np.cos(2*np.pi*t)])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coef, t, y

(a0, a1, a2, A, B), t_aj, y_aj = ajustar(AÑO_INICIO)

def tendencia(t):            # solo la parábola
    return a0 + a1*(t - 2000) + a2*(t - 2000)**2
def c(t):                    # modelo completo, en ppm
    return tendencia(t) + A*np.sin(2*np.pi*t) + B*np.cos(2*np.pi*t)
def dc(t):                   # derivada c'(t), en ppm/año
    return a1 + 2*a2*(t - 2000) + 2*np.pi*(A*np.cos(2*np.pi*t) - B*np.sin(2*np.pi*t))

res = y_aj - c(t_aj)
print(f"a0 = {a0:.3f}   a1 = {a1:.4f}   a2 = {a2:.6f}   A = {A:.4f}   B = {B:.4f}")
print(f"Error típico del ajuste (RMS): {np.sqrt(np.mean(res**2)):.2f} ppm   |  máximo: {np.abs(res).max():.2f} ppm")
print(f"Amplitud de la onda estacional: {np.hypot(A, B):.2f} ppm")
print(f"Parte estacional de c'(t): oscila entre ±{2*np.pi*np.hypot(A, B):.1f} ppm/año (¡la tendencia solo aporta ~3 ppm/año!)")

tt = np.linspace(AÑO_INICIO, 2036, 4000)
plt.figure(figsize=(9, 4))
plt.plot(datos.t, datos.co2, ".", ms=3, label="Mauna Loa (mensual)")
plt.plot(tt, c(tt), lw=1, label="modelo c(t)")
plt.plot(tt, tendencia(tt), "--", lw=1, label="solo tendencia")
plt.axhline(450, color="k", lw=0.8); plt.axvline(datos.t.iloc[-1], color="gray", ls=":")
plt.xlabel("año"); plt.ylabel("CO₂ (ppm)"); plt.legend(); plt.grid(alpha=.3); plt.show()

### Ítem II.1 · ¿En qué año cruza 450 ppm?
**Predice en tu guía antes de correr:** con la tendencia en el último dato (la imprime la celda) y su pendiente, ¿en qué año llegaría a 450 ppm si siguiera en línea recta? ¿El modelo completo cruzará **antes o después**? ¿Cuántas veces cruzará 450?

In [ ]:
# Celda 3 · Tendencia sola vs. modelo completo (todos los cruces entre 2026 y 2040)
t_ult = datos.t.iloc[-1]
pend = a1 + 2*a2*(t_ult - 2000)
print(f"Tendencia en el último dato (t = {t_ult}): {tendencia(t_ult):.2f} ppm,  pendiente {pend:.2f} ppm/año")
print(f"Recta (pendiente fija):   cruza en t ≈ {t_ult + (450 - tendencia(t_ult))/pend:.3f}")

def biseccion(f, a, b, tol=1e-11):
    fa = f(a)
    for _ in range(100):                 # 100 mitades bastan de sobra
        if b - a <= tol: break
        m = (a + b)/2
        if fa*f(m) <= 0: b = m
        else: a, fa = m, f(m)
    return (a + b)/2

MESES = ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"]
def fecha(t):
    return f"{MESES[min(int((t % 1)*12), 11)]} {int(t)}"

print(f"Parábola sola:            cruza en t ≈ {biseccion(lambda s: tendencia(s) - 450, 2026, 2040):.3f}")
malla = np.arange(2026.6, 2040, 1/365)          # un punto por día
f = c(malla) - 450
idx = np.where(np.sign(f[:-1]) != np.sign(f[1:]))[0]
cruces = [biseccion(lambda s: c(s) - 450, malla[i], malla[i+1]) for i in idx]
for k, r in enumerate(cruces, 1):
    print(f"Cruce {k} del modelo: t* = {r:.4f}  ({fecha(r)})   c'(t*) = {dc(r):6.2f} ppm/año")
t_estrella = cruces[0]

tt = np.linspace(2032.5, 2034.5, 1500)
plt.figure(figsize=(9, 3.5))
plt.plot(tt, c(tt), label="c(t)"); plt.plot(tt, tendencia(tt), "--", label="tendencia")
plt.axhline(450, color="k", lw=.8); plt.plot(cruces, [450]*len(cruces), "ro")
plt.xlabel("año"); plt.ylabel("ppm"); plt.legend(); plt.grid(alpha=.3); plt.show()

## Tus funciones
Completa los dos pasos (una línea cada uno). Si no lo logras, la celda de respaldo pone la versión correcta y te dice si la tuya coincide.

- Punto fijo con relajación: t_{n+1} = g(t_n) = t_n − (c(t_n) − 450)/K
- Secante: t_{k+1} = t_k − f(t_k)·(t_k − t_{k−1}) / (f(t_k) − f(t_{k−1})), con f(t) = c(t) − 450

In [ ]:
# Celda 4 · COMPLETA
def paso_punto_fijo(t, K):
    # devuelve g(t) = t - (c(t) - 450)/K
    return None   # <-- COMPLETA ESTO

def paso_secante(t_ant, t_act):
    # devuelve la raíz de la recta que pasa por (t_ant, c(t_ant)-450) y (t_act, c(t_act)-450)
    return None   # <-- COMPLETA ESTO

In [ ]:
# Celda 5 · Respaldo y verificación de tus funciones (no la cambies)
def _pf_ok(t, K):
    return t - (c(t) - 450)/K
def _sec_ok(t0, t1):
    f0, f1 = c(t0) - 450, c(t1) - 450
    if f1 == f0: return t1          # ya no se puede dividir: nos quedamos donde estamos
    return t1 - f1*(t1 - t0)/(f1 - f0)

if paso_punto_fijo(2033.0, 20) is None:
    paso_punto_fijo = _pf_ok; print("Punto fijo: usando la versión de respaldo.")
else:
    print("Tu punto fijo coincide:", bool(np.isclose(paso_punto_fijo(2033.0, 20), _pf_ok(2033.0, 20))))
if paso_secante(2032.9, 2033.0) is None:
    paso_secante = _sec_ok; print("Secante: usando la versión de respaldo.")
else:
    print("Tu secante coincide:", bool(np.isclose(paso_secante(2032.9, 2033.0), _sec_ok(2032.9, 2033.0))))

def punto_fijo(K, t0, tol=1e-8, nmax=500):
    """Itera hasta |t_{n+1} - t_n| < tol. Devuelve (lista de iterados, n de iteraciones o None)."""
    it = [t0]
    for n in range(1, nmax + 1):
        it.append(paso_punto_fijo(it[-1], K))
        if abs(it[-1]) > 1e5: return it, None            # se fue lejísimos
        if abs(it[-1] - it[-2]) < tol: return it, n
    return it, None

def secante(t0, t1, tol=1e-10, nmax=100):
    it = [t0, t1]
    for n in range(1, nmax + 1):
        if c(it[-1]) == c(it[-2]): return it, n - 1      # evita dividir 0/0
        it.append(paso_secante(it[-2], it[-1]))
        if abs(it[-1] - it[-2]) < tol: return it, n
    return it, None

def nombre_cruce(r):
    if r is None or not np.isfinite(r): return "ninguno"
    d = [abs(r - x) for x in cruces]
    return ["feb", "jul", "dic"][int(np.argmin(d))] + " 2033" if min(d) < 2e-3 else f"otro (t={r:.3f})"
print("Listo.")

### Ítem II.2 · ¿Converge el punto fijo?
Recuerda: g′(t) = 1 − c′(t)/K. La celda 3 te dio c′(t*) en el primer cruce.

**Predice en tu guía antes de correr:** para K = 10, K = 20 y K = 40, calcula |g′(t*)| y di si el punto fijo converge, y cuál será el más rápido.

In [ ]:
# Celda 6 · Probar valores de K desde t0 = 2033,0
T0 = 2033.0
VALORES_K = [10, 20, 40]          # <-- CAMBIA ESTO si quieres probar otros
cp = dc(t_estrella)
print(f"c'(t*) = {cp:.4f} ppm/año\n")
print("   K    g'(t*)   ¿convergió?  iteraciones   último iterado")
for K in VALORES_K:
    it, n = punto_fijo(K, T0)
    print(f"{K:5.1f}  {1 - cp/K:7.3f}   {'sí' if n else 'NO':^11}  {str(n):^11}   {it[-1]:.6f}")

it10, _ = punto_fijo(10, T0)
print("\nK = 10, últimos 4 iterados:", np.round(it10[-4:], 4))
plt.figure(figsize=(8, 3))
plt.plot(it10[:80], ".-", lw=.6); plt.axhline(t_estrella, color="r", lw=.8)
plt.title("Punto fijo con K = 10 (80 primeras iteraciones)"); plt.xlabel("n"); plt.ylabel("t_n"); plt.grid(alpha=.3); plt.show()

### Ítem II.3 · Tabla de errores: punto fijo vs. secante
**Predice en tu guía antes de correr:** para K = 40, ¿por cuánto se multiplica el error en cada paso? ¿Y para K = 20? Para la secante, ¿qué orden esperas leer en la última columna?

In [ ]:
# Celda 7 · Tablas de errores e_n = |t_n - t*|
def tabla_pf(K, t0=2033.0, filas=9):
    it, _ = punto_fijo(K, t0, tol=0, nmax=filas)
    e = [abs(x - t_estrella) for x in it]
    print(f"Punto fijo, K = {K}   (|g'(t*)| = {abs(1 - cp/K):.3f})")
    print("  n      e_n        e_n+1/e_n")
    for n in range(len(e) - 1):
        r = e[n+1]/e[n] if e[n] > 1e-12 and e[n+1] > 1e-12 else float("nan")
        print(f"{n:3d}  {e[n]:.3e}    {r:.3f}")
    print()

for K in [40, 20]:          # <-- CAMBIA ESTO (puedes agregar cp, que es c'(t*) exacto)
    tabla_pf(K)

it, _ = secante(2032.9, 2033.0, tol=0, nmax=6)
e = [abs(x - t_estrella) for x in it]
print("Secante desde t0 = 2032,9 y t1 = 2033,0")
print("  k      e_k       orden estimado p_k = log(e_k+1/e_k)/log(e_k/e_k-1)")
ordenes = []
for k in range(len(e)):
    if 1 <= k < len(e) - 1 and min(e[k-1], e[k], e[k+1]) > 1e-10:   # por debajo de 1e-10 manda el redondeo
        p = np.log(e[k+1]/e[k])/np.log(e[k]/e[k-1]); ordenes.append(p)
        print(f"{k:3d}  {e[k]:.3e}        {p:.2f}")
    else:
        print(f"{k:3d}  {e[k]:.3e}")
print(f"Promedio de los órdenes estimados (sin el primero): {np.mean(ordenes[1:]):.2f}")
print("\nRazón áurea (1+√5)/2 =", round((1 + 5**0.5)/2, 3))

### Ítem II.4 · ¿Qué cruce encuentra cada método?
Hay tres cruces cerca (feb, jul y dic de 2033).

**Predice en tu guía antes de correr:** el punto fijo con K = 20 arrancando en t₀ = 2033,7 (¡entre jul y dic!) ¿a qué cruce va? ¿Puede el punto fijo con algún K > 0 encontrar el cruce de **julio**? (Pista: mira el signo de c′ en julio.) ¿Y la secante?

In [ ]:
# Celda 8 · Qué cruce encuentra cada método según el arranque
K = 20                                               # <-- CAMBIA ESTO
for r in cruces:
    print(f"En {nombre_cruce(r)}: g'(t*) = 1 - c'(t*)/K = {1 - dc(r)/K:6.3f}")
print("\nPunto fijo (K = %g)" % K)
for t0 in [t_ult, 2030.0, 2033.4, 2033.7, 2035.0]:  # <-- CAMBIA ESTO
    it, n = punto_fijo(K, t0)
    print(f"  t0 = {t0:9.3f} -> {nombre_cruce(it[-1] if n else None):10s} ({n} iteraciones)")
print("\nSecante")
for par in [(t_ult, 2028.0), (t_ult, 2029.0), (t_ult, 2040.0)]:   # <-- CAMBIA ESTO
    it, n = secante(*par)
    print(f"  (t0, t1) = ({par[0]:.3f}, {par[1]:.1f}) -> {nombre_cruce(it[-1] if n else None):10s} ({n} iteraciones)   primeros: {np.round(it[2:5], 3)}")

### Ítem II.5 · Precisión de "un día"
Un día es 1/365 ≈ 0,00274 años. **Predice en tu guía antes de correr:** arrancando en t₀ = 2033,0 (error inicial ≈ 0,105), ¿cuántas iteraciones necesita el punto fijo con K = 40 para que el error sea menor que un día? (usa e_n ≈ 0,105·|g′(t*)|ⁿ). ¿Y la secante?

In [ ]:
# Celda 9 · Iteraciones hasta error < 1 día
UN_DIA = 1/365
def iter_hasta(it, tol=UN_DIA):
    for n, x in enumerate(it):
        if abs(x - t_estrella) < tol: return n
for K in [11, 20, 40]:                               # <-- CAMBIA ESTO
    it, _ = punto_fijo(K, 2033.0, tol=0, nmax=300)
    print(f"Punto fijo K = {K:3d}: {iter_hasta(it)} iteraciones")
it, _ = secante(2032.9, 2033.0, tol=0, nmax=10)
print(f"Secante (2032,9; 2033,0): {iter_hasta(it[1:])} iteraciones (sin contar t1)")

### Ítem II.6 · ¿Cuánto depende el cruce de la ventana de datos?
**Predice en tu guía antes de correr:** si ajustamos el modelo con datos desde 1960 (en vez de 1990), ¿el primer cruce se mueve días, meses o años?

In [ ]:
# Celda 10 · Sensibilidad a la ventana del ajuste (no cambia el modelo de las demás celdas)
for anio in [1960, 1980, 1990, 2000, 2010]:          # <-- CAMBIA ESTO
    (b0, b1, b2, BA, BB), _, _ = ajustar(anio)
    cc = lambda s: b0 + b1*(s-2000) + b2*(s-2000)**2 + BA*np.sin(2*np.pi*s) + BB*np.cos(2*np.pi*s) - 450
    m = np.arange(2026.6, 2040, 1/365); fm = cc(m)
    i = np.where(np.sign(fm[:-1]) != np.sign(fm[1:]))[0][0]
    r = biseccion(cc, m[i], m[i+1])
    print(f"Datos desde {anio}: primer cruce t ≈ {r:.3f} ({fecha(r)})")

## Parte III · Herramienta para auditar al agente
Usa esta celda para verificar (o refutar) las afirmaciones del agente. Cambia K, t₀ y el criterio de parada.

In [ ]:
# Celda 11 · Auditar al agente
K_AG = 20.45       # <-- CAMBIA ESTO
T0_AG = 2033.0     # <-- CAMBIA ESTO
CRITERIO = "paso"  # <-- CAMBIA ESTO: "paso" (|t_n+1 - t_n| < 1e-8, agente A) o "residuo" (|c(t_n) - 450| < 0,01 ppm, agente B)
NMAX = 500         # <-- CAMBIA ESTO: tope de iteraciones (el agente A usó 500)

t, n, hist = T0_AG, None, [T0_AG]
for k in range(1, NMAX + 1):
    t_nuevo = t - (c(t) - 450)/K_AG
    hist.append(t_nuevo)
    if abs(t_nuevo) > 1e5: break
    para = abs(t_nuevo - t) < 1e-8 if CRITERIO == "paso" else abs(c(t) - 450) < 0.01
    if para:
        n = k if CRITERIO == "paso" else k - 1
        t = t_nuevo if CRITERIO == "paso" else t; break
    t = t_nuevo
print(f"K = {K_AG}, t0 = {T0_AG}, criterio '{CRITERIO}', tope {NMAX}: iteraciones = {n}" + ("  (llegó al tope o divergió)" if n is None else ""))
print(f"Resultado: t = {t:.5f}   c(t) = {c(t):.4f} ppm   ¿cruce conocido? {nombre_cruce(t if n is not None else None)}")
print(f"g'(t*) en el primer cruce = 1 - c'(t*)/K = {1 - dc(t_estrella)/K_AG:.3f}   |   g' en el último t = {1 - dc(t)/K_AG:.3f}")
if not (t_aj[0] <= t <= t_aj[-1] + 10):
    print(f"OJO: t = {t:.2f} está lejos del rango de datos del ajuste [{t_aj[0]:.1f}, {t_aj[-1]:.1f}]")
print("Primeros iterados:", np.round(hist[:6], 4))

## Parte I · Comprobación de las hipótesis del teorema (opcional)

In [ ]:
# Celda 12 · ¿Cumple g(t) = t - (c(t) - 450)/K las hipótesis del teorema en [a, b]?
a_, b_, K_ = 2033.0, 2033.2, 20     # <-- CAMBIA ESTO
m = np.linspace(a_, b_, 200001)
g_m = m - (c(m) - 450)/K_
gp_m = 1 - dc(m)/K_
print(f"g([{a_}, {b_}]) está dentro de [{g_m.min():.4f}, {g_m.max():.4f}]  -> ¿dentro de [a, b]? {a_ <= g_m.min() and g_m.max() <= b_}")
print(f"máx |g'(t)| en [a, b] = {np.abs(gp_m).max():.3f}")

---
**Recuerda:** todo lo que calculaste es el cruce del **modelo** que ajustamos (una parábola más una onda). No es una predicción climática oficial: las proyecciones serias usan escenarios de emisiones y modelos del ciclo del carbono. Escribe tu reflexión de una línea en la guía (ítem II.6).